In [ ]:
import os
from dotenv import find_dotenv, load_dotenv
from langchain.chat_models import init_chat_model
from langchain.agents import create_agent
from langchain.messages import HumanMessage, AIMessage, RemoveMessage
from pydantic import BaseModel
from langchain.tools import tool
from pydantic import BaseModel, Field
# 使用tavily作为web搜索工具
from langchain_tavily import TavilySearch, tavily_crawl
from langchain.agents.middleware import SummarizationMiddleware
from langgraph.checkpoint.memory import InMemorySaver
from langchain_core.runnables import RunnableConfig

from langgraph.graph.message import REMOVE_ALL_MESSAGES
from langgraph.checkpoint.memory import InMemorySaver
from langchain.agents import create_agent, AgentState
from langchain.agents.middleware import before_model
from langgraph.runtime import Runtime
from langchain_core.runnables import RunnableConfig
from typing import Any
from langchain.messages import AnyMessage
from typing import TypedDict

load_dotenv(find_dotenv())

api_key = os.getenv("DASHSCOPE_API_KEY")
base_url = os.getenv("DASHSCOPE_BASE_URL")



In [2]:
model = init_chat_model(
    model="qwen3.5-35b-a3b",
    model_provider="openai",
    api_key=api_key,
    base_url=base_url,
    # 尝试显式关闭思考模式
    extra_body={"enable_thinking": False} 
)

In [8]:
@before_model
def trim_messages(state: AgentState, runtime: Runtime) -> dict[str, Any] | None:
    """Keep only the last few messages to fit context window."""
    messages = state["messages"]

    if len(messages) <= 3:
        return None
    
    first_msg = messages[0]
    
    recent_messages = messages[-3:] if len(messages) % 2 == 0 else messages[-4:]

    new_messages = [first_msg] + recent_messages

    return {
        "messages": [
            RemoveMessage(id=REMOVE_ALL_MESSAGES),
            *new_messages
        ]
    }

In [9]:
agent = create_agent(
    model=model,
    middleware=[trim_messages],
    checkpointer=InMemorySaver()
)

In [11]:
config: RunnableConfig = {"configurable": {"thread_id": "1"}}

agent.invoke({"messages": "你好我是jzl"}, config)
agent.invoke({"messages": "我最喜欢的运动是羽毛球"}, config)
agent.invoke({"messages": "我最喜欢的食物是火锅"}, config)

final_response = agent.invoke({"messages": "你还记得我吗"}, config)

final_response["messages"][-1].pretty_print()

================================== Ai Message ==================================

当然记得！你是 **Jzl**，一位热爱**羽毛球**（全能运动、反应快）且对**火锅**情有独钟的美食家。🏸🍲

刚才我们聊到了你的运动爱好和饮食偏好，这些细节我都记着呢！是不是觉得突然被“点名”有点小惊喜？😄

接下来你想聊聊什么？
*   是想继续深入探讨**羽毛球技巧**或装备？
*   还是想让我推荐几家适合打完球后去吃的**宝藏火锅店**？
*   或者只是随便聊聊其他话题？

随时听你吩咐！
